# Hand Gesture Recognition

**Task:** Develop a model that can accurately identify and classify different hand gestures from image data, enabling gesture-based human-computer interaction.

This notebook covers:
1. Load gesture images (with real-dataset loading code included)
2. Preprocess images
3. Extract features (HOG)
4. Split into train/test sets
5. Train and compare classifiers (SVM and Random Forest)
6. Evaluate performance (accuracy, confusion matrix, per-class report)
7. Predict on a new gesture image
8. Notes on extending to real-time video/webcam input


## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
IMG_SIZE = 64
GESTURES = ['palm', 'fist', 'thumbs_up', 'peace', 'ok']


## 2. Load the Dataset

**To use a real dataset**, the most common choice for this task is Kaggle's **LeapGestRecog** dataset
(https://www.kaggle.com/datasets/gti-upm/leapgestrecog), which has ~20,000 images across 10 gesture classes
performed by 10 different subjects.

After downloading and extracting it, the folder structure looks like:
```
leapGestRecog/
  00/
    01_palm/*.png
    02_l/*.png
    ...
  01/
    01_palm/*.png
    ...
```
Load it with:
```python
import cv2

def load_leapgestrecog(root_dir):
    images, labels = [], []
    for subject in sorted(os.listdir(root_dir)):
        subject_path = os.path.join(root_dir, subject)
        if not os.path.isdir(subject_path):
            continue
        for gesture_folder in sorted(os.listdir(subject_path)):
            gesture_path = os.path.join(subject_path, gesture_folder)
            label = gesture_folder.split('_', 1)[1]  # e.g. '01_palm' -> 'palm'
            for fname in os.listdir(gesture_path):
                img = cv2.imread(os.path.join(gesture_path, fname), cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(label)
    return images, labels

images, labels = load_leapgestrecog('leapGestRecog')
```

For now, we'll generate **synthetic grayscale gesture images** for 5 classes so the full pipeline runs end-to-end without downloading anything. The rest of the pipeline (feature extraction, training, evaluation) is identical either way — you're only swapping the data source.

In [ ]:
# --- Synthetic gesture image generator (replace with real dataset loading above) ---

def make_synthetic_gesture(label, size=IMG_SIZE):
    """Generates a simple synthetic grayscale image with a distinct shape per gesture class."""
    img = np.random.normal(30, 8, (size, size))  # dark background
    yy, xx = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2

    if label == 'palm':  # open circle (all fingers spread = big round blob)
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        img += np.where(dist < size * 0.38, 150, 0)
    elif label == 'fist':  # small compact blob
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        img += np.where(dist < size * 0.22, 150, 0)
    elif label == 'thumbs_up':  # blob + vertical bar on top (thumb)
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy * 1.4) ** 2)
        img += np.where(dist < size * 0.2, 150, 0)
        img[0:size // 2, cx - size // 10: cx + size // 10] += 130
    elif label == 'peace':  # blob + two vertical bars (fingers)
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy * 1.4) ** 2)
        img += np.where(dist < size * 0.2, 150, 0)
        img[0:size // 2, cx - size // 6: cx - size // 12] += 130
        img[0:size // 2, cx + size // 12: cx + size // 6] += 130
    elif label == 'ok':  # ring shape (thumb+index circle) + small blob
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        ring = (dist < size * 0.22) & (dist > size * 0.12)
        img += np.where(ring, 150, 0)

    img += np.random.normal(0, 8, (size, size))
    return np.clip(img, 0, 255).astype(np.uint8)

n_per_class = 150
images, labels = [], []

for gesture in GESTURES:
    for _ in range(n_per_class):
        images.append(make_synthetic_gesture(gesture))
        labels.append(gesture)

print(f'Total images: {len(images)}  |  Classes: {GESTURES}')


In [ ]:
# Preview one example per gesture class
fig, axes = plt.subplots(1, len(GESTURES), figsize=(15, 3))
for i, gesture in enumerate(GESTURES):
    idx = labels.index(gesture)
    axes[i].imshow(images[idx], cmap='gray')
    axes[i].set_title(gesture)
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## 3. Extract Features using HOG

As with static image classification tasks, we use **HOG (Histogram of Oriented Gradients)** to turn each image into a feature vector capturing hand shape/edge information — this works well with classical classifiers like SVM and Random Forest, and is fast to train without needing a GPU.

In [ ]:
X = np.array([hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)) for img in images])

le = LabelEncoder()
y = le.fit_transform(labels)

print(f'Feature matrix shape: {X.shape}')
print(f'Classes: {list(le.classes_)}')


## 4. Split into Train / Test Sets and Scale Features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')


## 5. Train and Compare Classifiers

We'll train both an SVM and a Random Forest, and compare their accuracy — a good practice so you can justify your final model choice.

In [ ]:
svm_model = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_acc = accuracy_score(y_test, svm_model.predict(X_test_scaled))

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))

print(f'SVM accuracy: {svm_acc:.4f}')
print(f'Random Forest accuracy: {rf_acc:.4f}')

best_model = svm_model if svm_acc >= rf_acc else rf_model
best_name = 'SVM' if svm_acc >= rf_acc else 'Random Forest'
print(f'\nUsing {best_name} as the final model.')


## 6. Evaluate the Final Model

In [ ]:
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix: Gesture Classification')
plt.tight_layout()
plt.show()


## 7. Predict on a New Gesture Image

In [ ]:
new_image = make_synthetic_gesture('peace')  # replace with a real loaded+resized+grayscaled image
new_features = hog(new_image, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)).reshape(1, -1)
new_features_scaled = scaler.transform(new_features)

prediction = best_model.predict(new_features_scaled)[0]
predicted_label = le.inverse_transform([prediction])[0]

plt.imshow(new_image, cmap='gray')
plt.title(f'Predicted gesture: {predicted_label}')
plt.axis('off')
plt.show()


## 8. Extending to Real-Time Video / Webcam Input

For true gesture-based human-computer interaction, you'd run this pipeline on frames from a live webcam feed. Here's the general structure (not run in this notebook, since it needs a live camera):

```python
import cv2

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))

    features = hog(resized, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)).reshape(1, -1)
    features_scaled = scaler.transform(features)
    pred = best_model.predict(features_scaled)[0]
    gesture_name = le.inverse_transform([pred])[0]

    cv2.putText(frame, gesture_name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Hand Gesture Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
```

For more robust real-time hand detection (isolating the hand from a cluttered background before classification), tools like **MediaPipe Hands** are commonly paired with this kind of pipeline to first detect and crop the hand region.

## Summary

> **Note:** As with other tasks using synthetic placeholder data, accuracy here is very high because the shapes are artificially distinct. On the real LeapGestRecog dataset (or your own webcam captures), expect accuracy to vary more — typically 85-98% is achievable with a well-tuned classical HOG + SVM/Random Forest pipeline on a controlled, well-lit dataset like LeapGestRecog, though results will be lower with cluttered/inconsistent backgrounds.

- We built a multi-class hand gesture classifier using **HOG features** with both **SVM** and **Random Forest**, and picked the better-performing model.
- We evaluated it with per-class precision/recall/F1 and a confusion matrix.
- We outlined how to extend the pipeline to real-time webcam-based gesture recognition.

**Next steps to make this more robust for a real submission:**
- Download and use the real LeapGestRecog dataset (or record your own gesture images/videos)
- Try a CNN (e.g. with TensorFlow/Keras or PyTorch) for potentially higher accuracy, especially on messier real-world images
- Add hand detection/cropping (e.g. MediaPipe Hands) before classification to handle cluttered backgrounds
- Tune hyperparameters (`C`, `gamma` for SVM; `n_estimators`, `max_depth` for Random Forest) with `GridSearchCV`
